In [98]:
# Data Manipulation Libraries
import pandas as pd
import numpy as np

# Data Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

# Preprocessing Libraries
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Deep Learning Libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# Model Evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Save Preprocessor
import pickle
import os

In [99]:
df = pd.read_csv("cost-analysis.csv")
df

,UsageDate,ServiceName,CostUSD,Cost,Currency
0,2023-05-01,Automation,0.001000,0.081771,INR
1,2023-05-01,Azure DNS,0.500348,40.914076,INR
2,2023-05-01,Bandwidth,127.770719,10447.971394,INR
3,2023-05-01,Storage,134.504502,10998.601241,INR
4,2023-05-01,Virtual Machines,13.987309,1143.759767,INR
...,...,...,...,...,...
84,2024-04-01,Logic Apps,0.003708,0.308808,INR
85,2024-04-01,Storage,152.335920,12688.249184,INR
86,2024-04-01,Virtual Machines,22.328693,1859.784719,INR
87,2024-04-01,Virtual Machines Licenses,27.547339,2294.452273,INR


In [100]:
df.head()

,UsageDate,ServiceName,CostUSD,Cost,Currency
0,2023-05-01,Automation,0.001000,0.081771,INR
1,2023-05-01,Azure DNS,0.500348,40.914076,INR
2,2023-05-01,Bandwidth,127.770719,10447.971394,INR
3,2023-05-01,Storage,134.504502,10998.601241,INR
4,2023-05-01,Virtual Machines,13.987309,1143.759767,INR


In [101]:
df.shape

(89, 5)

In [102]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89 entries, 0 to 88
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   UsageDate    89 non-null     object 
 1   ServiceName  89 non-null     object 
 2   CostUSD      89 non-null     float64
 3   Cost         89 non-null     float64
 4   Currency     89 non-null     object 
dtypes: float64(2), object(3)
memory usage: 3.6+ KB


In [103]:
df.describe()

,CostUSD,Cost
count,89.000000,89.000000
mean,47.157293,3905.133202
std,56.253490,4658.406713
min,0.000000,0.000000
25%,0.484221,40.309546
50%,23.789508,1952.895445
75%,123.555142,10138.471661
max,154.816673,12887.907485


In [104]:
df.isnull().sum()

UsageDate      0
ServiceName    0
CostUSD        0
Cost           0
Currency       0
dtype: int64

In [105]:
df.duplicated().sum()

np.int64(0)

In [106]:
df.columns

Index(['UsageDate', 'ServiceName', 'CostUSD', 'Cost', 'Currency'], dtype='object')

In [107]:
df.dtypes

UsageDate       object
ServiceName     object
CostUSD        float64
Cost           float64
Currency        object
dtype: object

In [108]:
df["UsageDate"] = pd.to_datetime(df["UsageDate"])

In [109]:
df["Year"] = df["UsageDate"].dt.year
df["Month"] = df["UsageDate"].dt.month
df["Day"] = df["UsageDate"].dt.day

In [110]:
df = df.drop("UsageDate",axis=1)

In [111]:
X = df.drop("Cost", axis=1)
y = df["Cost"]

In [112]:
num_cols=X.select_dtypes(exclude="object").columns
cat_cols=X.select_dtypes(include="object").columns

In [113]:
print(num_cols)
print(cat_cols)

Index(['CostUSD', 'Year', 'Month', 'Day'], dtype='object')
Index(['ServiceName', 'Currency'], dtype='object')


In [114]:
num_pipeline = Pipeline(steps=[("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler())])

In [115]:
cat_pipeline = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("onehotencoder",OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [116]:
preprocessor = ColumnTransformer(
    [
        ("num_pipeline",num_pipeline,num_cols),
        ("cat_pipeline",cat_pipeline,cat_cols)
    ]
)

In [117]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [118]:
print(X_train.columns)

Index(['ServiceName', 'CostUSD', 'Currency', 'Year', 'Month', 'Day'], dtype='object')


In [119]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [120]:
with open("preprocessor.pkl","wb") as file:
    pickle.dump(preprocessor,file)

In [121]:
model = Sequential()
model.add(Dense(128, activation="relu", input_dim=X_train.shape[1]))
model.add(Dense(64, activation="relu"))
model.add(Dense(32, activation="relu"))
model.add(Dense(1, activation="linear"))

In [122]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 128)                 │           1,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 12,161 (47.50 KB)

 Trainable params: 12,161 (47.50 KB)

 Non-trainable params: 0 (0.00 B)

In [123]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [126]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)

In [127]:
history = model.fit(X_train,y_train,validation_split=0.2,epochs=100,batch_size=32,callbacks=[early_stopping],verbose=1)

Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 372ms/step - loss: 42724992.0000 - mae: 4426.2397 - val_loss: 29020968.0000 - val_mae: 3511.4949
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - loss: 42724160.0000 - mae: 4426.1543 - val_loss: 29020380.0000 - val_mae: 3511.4307
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - loss: 42723448.0000 - mae: 4426.0942 - val_loss: 29019924.0000 - val_mae: 3511.3772
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 42722900.0000 - mae: 4426.0474 - val_loss: 29019478.0000 - val_mae: 3511.3279
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - loss: 42722232.0000 - mae: 4425.9917 - val_loss: 29018966.0000 - val_mae: 3511.2727
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 42721528.0000 - mae: 4425.9346 - val_loss: 29018432.0000 - val_mae: 3511.2104
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - loss: 42720740.0000 - mae: 4425.8716 - val_loss: 29017844.0000 - val_mae: 3511.1516
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/

In [128]:
loss, mae = model.evaluate(X_test, y_test)

print("Loss :", loss)
print("MAE :", mae)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 12159334.0000 - mae: 2260.8611
Loss : 12159334.0
MAE : 2260.861083984375


In [129]:
model.save("model.keras")